In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — INSTALL LIBRARIES                                      ║
# ╚══════════════════════════════════════════════════════════════════╝

import subprocess, sys
for lib in ["kagglehub","catboost","shap","optuna",
            "vaderSentiment","yake","textstat"]:
    subprocess.run([sys.executable,"-m","pip","install",lib,"-q"])
print("All libraries installed.")


All libraries installed.


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — GOOGLE DRIVE                                           ║
# ╚══════════════════════════════════════════════════════════════════╝

import os, shutil
from google.colab import drive, files

drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/RealEstate_TXNY"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

Mounted at /content/drive
Save directory: /content/drive/MyDrive/RealEstate_TXNY


In [3]:
# ╔════════════════════════════════════════════════════╗
# ║  CELL 3 — IMPORT ALL LIBRARIES                      ║
# ╚═════════════════════════════════════════════════════╝

import os
import re
import warnings
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn
from sklearn.model_selection import train_test_split, KFold, cross_val_predict, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, f1_score, classification_report,
    accuracy_score,
)

# Gradient boosting
!pip install catboost
from catboost import CatBoostRegressor, CatBoostClassifier
import xgboost as xgb

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Hyperparameter tuning
!pip install optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Interpretability
import shap

# NLP
!pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
!pip install yake
import yake
!pip install textstat
import textstat

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.4f}".format)

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 60)
print("All libraries imported successfully.")
print(f"  pandas     : {pd.__version__}")
print(f"  numpy      : {np.__version__}")
print(f"  tensorflow : {tf.__version__}")
print(f"  sklearn    : OK")
print("=" * 60)

All libraries imported successfully.
  pandas     : 2.2.2
  numpy      : 2.0.2
  tensorflow : 2.20.0
  sklearn    : OK


In [4]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — STATE NAME NORMALISATION                               ║
# ╚══════════════════════════════════════════════════════════════════╝
# SAKIB uses full state names: "Texas", "New York" etc.
# After .str.upper() → "TEXAS", "NEW YORK" which do NOT match "TX", "NY"
# This dictionary converts every full name to its 2-letter abbreviation.
# Applied BEFORE any filtering so no rows are lost.

STATE_MAP = {
    "ALABAMA":"AL","ALASKA":"AK","ARIZONA":"AZ","ARKANSAS":"AR",
    "CALIFORNIA":"CA","COLORADO":"CO","CONNECTICUT":"CT","DELAWARE":"DE",
    "FLORIDA":"FL","GEORGIA":"GA","HAWAII":"HI","IDAHO":"ID",
    "ILLINOIS":"IL","INDIANA":"IN","IOWA":"IA","KANSAS":"KS",
    "KENTUCKY":"KY","LOUISIANA":"LA","MAINE":"ME","MARYLAND":"MD",
    "MASSACHUSETTS":"MA","MICHIGAN":"MI","MINNESOTA":"MN","MISSISSIPPI":"MS",
    "MISSOURI":"MO","MONTANA":"MT","NEBRASKA":"NE","NEVADA":"NV",
    "NEW HAMPSHIRE":"NH","NEW JERSEY":"NJ","NEW MEXICO":"NM","NEW YORK":"NY",
    "NORTH CAROLINA":"NC","NORTH DAKOTA":"ND","OHIO":"OH","OKLAHOMA":"OK",
    "OREGON":"OR","PENNSYLVANIA":"PA","RHODE ISLAND":"RI","SOUTH CAROLINA":"SC",
    "SOUTH DAKOTA":"SD","TENNESSEE":"TN","TEXAS":"TX","UTAH":"UT",
    "VERMONT":"VT","VIRGINIA":"VA","WASHINGTON":"WA","WEST VIRGINIA":"WV",
    "WISCONSIN":"WI","WYOMING":"WY","DISTRICT OF COLUMBIA":"DC",
    # Already-abbreviated pass through
    "AL":"AL","AK":"AK","AZ":"AZ","AR":"AR","CA":"CA","CO":"CO",
    "CT":"CT","DE":"DE","FL":"FL","GA":"GA","HI":"HI","ID":"ID",
    "IL":"IL","IN":"IN","IA":"IA","KS":"KS","KY":"KY","LA":"LA",
    "ME":"ME","MD":"MD","MA":"MA","MI":"MI","MN":"MN","MS":"MS",
    "MO":"MO","MT":"MT","NE":"NE","NV":"NV","NH":"NH","NJ":"NJ",
    "NM":"NM","NY":"NY","NC":"NC","ND":"ND","OH":"OH","OK":"OK",
    "OR":"OR","PA":"PA","RI":"RI","SC":"SC","SD":"SD","TN":"TN",
    "TX":"TX","UT":"UT","VT":"VT","VA":"VA","WA":"WA","WV":"WV",
    "WI":"WI","WY":"WY","DC":"DC",
}

def normalise_state(series):
    """Convert state column to 2-letter uppercase abbreviations."""
    return (series.astype(str).str.strip().str.upper()
            .map(lambda x: STATE_MAP.get(x, x)))

# Quick test
test = pd.Series(["Texas","New York","CA","florida","OHIO","TX","ny"])
result = normalise_state(test)
print("State normalisation test:")
for raw, norm in zip(test, result):
    print(f"  '{raw}' → '{norm}'")

State normalisation test:
  'Texas' → 'TX'
  'New York' → 'NY'
  'CA' → 'CA'
  'florida' → 'FL'
  'OHIO' → 'OH'
  'TX' → 'TX'
  'ny' → 'NY'


In [5]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — DOWNLOAD ALL 4 DATASETS                                ║
# ╚══════════════════════════════════════════════════════════════════╝

print("Downloading 4 datasets ...")
path_sakib    = kagglehub.dataset_download("ahmedshahriarsakib/usa-real-estate-dataset")
path_polartech= kagglehub.dataset_download("polartech/500000-us-homes-data-for-sale-properties")
path_texas    = kagglehub.dataset_download("jahnavikachhia23/texas-residential-real-estate-intelligence-2026")
path_newyork  = kagglehub.dataset_download("kanchana1990/new-york-real-estate-data-2026")
print(f"  SAKIB     : {path_sakib}")
print(f"  POLARTECH : {path_polartech}")
print(f"  TEXAS     : {path_texas}")
print(f"  NEW YORK  : {path_newyork}")

Using Colab cache for faster access to the 'usa-real-estate-dataset' dataset.


100%|██████████| 34.6M/34.6M [00:02<00:00, 17.1MB/s]

Extracting files...


100%|██████████| 3.46M/3.46M [00:01<00:00, 3.56MB/s]

Extracting files...


100%|██████████| 3.11M/3.11M [00:00<00:00, 3.28MB/s]

Extracting files...
  SAKIB     : /kaggle/input/usa-real-estate-dataset
  POLARTECH : /root/.cache/kagglehub/datasets/polartech/500000-us-homes-data-for-sale-properties/versions/1
  TEXAS     : /root/.cache/kagglehub/datasets/jahnavikachhia23/texas-residential-real-estate-intelligence-2026/versions/1
  NEW YORK  : /root/.cache/kagglehub/datasets/kanchana1990/new-york-real-estate-data-2026/versions/1


In [6]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — LOAD ALL 4 CSV FILES                                   ║
# ╚══════════════════════════════════════════════════════════════════╝

def find_csv(folder):
    """Return path of first CSV found in folder tree."""
    for root, dirs, fs in os.walk(folder):
        for f in fs:
            if f.endswith(".csv"):
                return os.path.join(root, f)
    raise FileNotFoundError(f"No CSV in: {folder}")

df_sakib     = pd.read_csv(find_csv(path_sakib),     low_memory=False)
df_polartech = pd.read_csv(find_csv(path_polartech), low_memory=False)
df_texas     = pd.read_csv(find_csv(path_texas),     low_memory=False)
df_newyork   = pd.read_csv(find_csv(path_newyork),   low_memory=False)

for name, df in [("SAKIB",df_sakib),("POLARTECH",df_polartech),
                  ("TEXAS",df_texas),("NEW YORK",df_newyork)]:
    print(f"{name:10s}: {df.shape[0]:,} rows x {df.shape[1]} cols | cols: {list(df.columns)}")


SAKIB     : 2,226,382 rows x 12 cols | cols: ['brokered_by', 'status', 'price', 'bed', 'bath', 'acre_lot', 'street', 'city', 'state', 'zip_code', 'house_size', 'prev_sold_date']
POLARTECH : 600,000 rows x 28 cols | cols: ['property_url', 'property_id', 'address', 'street_name', 'apartment', 'city', 'state', 'latitude', 'longitude', 'postcode', 'price', 'bedroom_number', 'bathroom_number', 'price_per_unit', 'living_space', 'land_space', 'land_space_unit', 'broker_id', 'property_type', 'property_status', 'year_build', 'total_num_units', 'listing_age', 'RunDate', 'agency_name', 'agent_name', 'agent_phone', 'is_owned_by_zillow']
TEXAS     : 12,137 rows x 13 cols | cols: ['type', 'sub_type', 'text', 'listPrice', 'sqft', 'stories', 'beds', 'baths', 'baths_full', 'baths_full_calc', 'garage', 'year_built', 'Price_Per_SqFt']
NEW YORK  : 8,273 rows x 11 cols | cols: ['type', 'sub_type', 'text', 'listPrice', 'sqft', 'stories', 'beds', 'baths', 'baths_full', 'baths_full_calc', 'garage']


In [7]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — INSPECT TEXAS AND NEW YORK DESCRIPTION COLUMNS         ║
# ╚══════════════════════════════════════════════════════════════════╝

TEXAS_DESC   = "description"
NEWYORK_DESC = "description"

for name, df, dc in [("TEXAS",df_texas,TEXAS_DESC),
                      ("NEW YORK",df_newyork,NEWYORK_DESC)]:
    print(f"\n{'='*55}")
    print(f"{name} — all columns: {list(df.columns)}")
    if dc in df.columns:
        valid = df[dc].dropna()
        print(f"  '{dc}': {len(valid):,} rows, avg {valid.str.len().mean():.0f} chars")
        for i, d in enumerate(valid.head(2)):
            print(f"  Sample [{i+1}]: {str(d)[:200]}")
    else:
        cands = [c for c in df.columns if any(kw in c.lower()
                 for kw in ["desc","text","remark","detail","note"])]
        print(f"  '{dc}' NOT found. Candidates: {cands}")
        print(f"  Update TEXAS_DESC or NEWYORK_DESC above.")



TEXAS — all columns: ['type', 'sub_type', 'text', 'listPrice', 'sqft', 'stories', 'beds', 'baths', 'baths_full', 'baths_full_calc', 'garage', 'year_built', 'Price_Per_SqFt']
  'description' NOT found. Candidates: ['text']
  Update TEXAS_DESC or NEWYORK_DESC above.

NEW YORK — all columns: ['type', 'sub_type', 'text', 'listPrice', 'sqft', 'stories', 'beds', 'baths', 'baths_full', 'baths_full_calc', 'garage']
  'description' NOT found. Candidates: ['text']
  Update TEXAS_DESC or NEWYORK_DESC above.


In [8]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — CLEAN SAKIB                                            ║
# ╚══════════════════════════════════════════════════════════════════╝

df_sakib_clean = df_sakib.copy()
df_sakib_clean.columns = (df_sakib_clean.columns.str.strip().str.lower()
                           .str.replace(r"[\s\-]+","_",regex=True))

# Detect column names
BED_COL  = "bed"        if "bed"        in df_sakib_clean.columns else "beds"
BATH_COL = "bath"       if "bath"       in df_sakib_clean.columns else "baths"
SQFT_COL = "house_size" if "house_size" in df_sakib_clean.columns else "sqfoot"
CITY_COL = "city"       if "city"       in df_sakib_clean.columns else None
print(f"SAKIB: bed='{BED_COL}' bath='{BATH_COL}' sqft='{SQFT_COL}' city='{CITY_COL}'")

# ── APPLY STATE FIX FIRST before any filtering ───────────────────
print(f"\nBefore state fix — TX: {(df_sakib_clean['state'].str.upper()=='TX').sum():,}  "
      f"NY: {(df_sakib_clean['state'].str.upper()=='NY').sum():,}")
df_sakib_clean["state"] = normalise_state(df_sakib_clean["state"])
print(f"After  state fix — TX: {(df_sakib_clean['state']=='TX').sum():,}  "
      f"NY: {(df_sakib_clean['state']=='NY').sum():,}  ← should be > 0")

# Standard cleaning
n = len(df_sakib_clean)
df_sakib_clean = df_sakib_clean.drop_duplicates()
df_sakib_clean = df_sakib_clean.dropna(subset=["price","state"])
print(f"After dedup+dropna: {len(df_sakib_clean):,} (removed {n-len(df_sakib_clean):,})")

q1 = df_sakib_clean["price"].quantile(0.25)
q3 = df_sakib_clean["price"].quantile(0.75)
n = len(df_sakib_clean)
df_sakib_clean = df_sakib_clean[
    (df_sakib_clean["price"] >= q1-1.5*(q3-q1)) &
    (df_sakib_clean["price"] <= q3+1.5*(q3-q1))]
print(f"After IQR clip: {len(df_sakib_clean):,}")

df_sakib_clean["log_price"] = np.log1p(df_sakib_clean["price"])

safe = [c for c in [BED_COL,BATH_COL,SQFT_COL,"acre_lot"]
        if c in df_sakib_clean.columns]
mice = IterativeImputer(max_iter=10,random_state=SEED)
df_sakib_clean[safe] = mice.fit_transform(df_sakib_clean[safe])
df_sakib_clean[BED_COL]  = df_sakib_clean[BED_COL].clip(0,20)
df_sakib_clean[BATH_COL] = df_sakib_clean[BATH_COL].clip(0,20)
if CITY_COL:
    df_sakib_clean[CITY_COL] = df_sakib_clean[CITY_COL].astype(str).str.strip().str.upper()
df_sakib_clean["source"] = "sakib"

print(f"\nSAKIB clean: {df_sakib_clean.shape[0]:,} rows")
print(f"  TX rows: {(df_sakib_clean['state']=='TX').sum():,}")
print(f"  NY rows: {(df_sakib_clean['state']=='NY').sum():,}")

if (df_sakib_clean["state"]=="TX").sum()==0:
    raise ValueError("TX still 0 after state fix! Print df_sakib raw state values above.")
if (df_sakib_clean["state"]=="NY").sum()==0:
    raise ValueError("NY still 0 after state fix! Print df_sakib raw state values above.")



SAKIB: bed='bed' bath='bath' sqft='house_size' city='city'

Before state fix — TX: 0  NY: 0
After  state fix — TX: 208,335  NY: 103,159  ← should be > 0
After dedup+dropna: 2,224,841 (removed 1,541)
After IQR clip: 2,053,241

SAKIB clean: 2,053,241 rows
  TX rows: 197,964
  NY rows: 84,399


In [9]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — CLEAN POLARTECH                                        ║
# ╚══════════════════════════════════════════════════════════════════╝

df_polartech_clean = df_polartech.copy()
df_polartech_clean.columns = (df_polartech_clean.columns.str.strip().str.lower()
                               .str.replace(r"[\s\-]+","_",regex=True))

print(f"POLARTECH before fix — TX: {(df_polartech_clean['state'].astype(str).str.upper()=='TX').sum():,}")
df_polartech_clean["state"] = normalise_state(df_polartech_clean["state"])
print(f"POLARTECH after  fix — TX: {(df_polartech_clean['state']=='TX').sum():,}  "
      f"NY: {(df_polartech_clean['state']=='NY').sum():,}")

n = len(df_polartech_clean)
df_polartech_clean = df_polartech_clean.drop_duplicates()
df_polartech_clean = df_polartech_clean.dropna(subset=["price","state"])
q1p=df_polartech_clean["price"].quantile(0.25); q3p=df_polartech_clean["price"].quantile(0.75)
iqrp=q3p-q1p
df_polartech_clean=df_polartech_clean[
    (df_polartech_clean["price"]>=q1p-1.5*iqrp)&
    (df_polartech_clean["price"]<=q3p+1.5*iqrp)]
df_polartech_clean["log_price"]=np.log1p(df_polartech_clean["price"])
if "city" in df_polartech_clean.columns:
    df_polartech_clean["city"]=df_polartech_clean["city"].astype(str).str.strip().str.upper()
df_polartech_clean["source"]="polartech"
print(f"POLARTECH clean: {df_polartech_clean.shape[0]:,} rows  "
      f"TX={( df_polartech_clean['state']=='TX').sum():,}  "
      f"NY={(df_polartech_clean['state']=='NY').sum():,}")

POLARTECH before fix — TX: 146,636
POLARTECH after  fix — TX: 146,636  NY: 0
POLARTECH clean: 554,979 rows  TX=139,699  NY=0


In [10]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — MERGE AND EXTRACT TX + NY ROWS ONLY                   ║
# ╚══════════════════════════════════════════════════════════════════╝
# Merge SAKIB + POLARTECH, then immediately extract ONLY TX and NY rows.
# Every row will have a real NLP value

import gc # Import the garbage collector module

TARGET_MIN = 200_000
TARGET_MAX = 300_000

df_primary = pd.concat([df_sakib_clean,df_polartech_clean],
                        axis=0,ignore_index=True,sort=False)
dedup_key = ["price","state"]+[c for c in ["bed","beds","bath","baths","city"]
                                 if c in df_primary.columns]
n = len(df_primary)
df_primary = df_primary.drop_duplicates(subset=dedup_key,keep="first")
print(f"Primary corpus after merge+dedup: {df_primary.shape[0]:,} rows")

# Extract ONLY TX and NY rows
df_txny = df_primary[df_primary["state"].isin(["TX","NY"])].copy()
n_tx = (df_txny["state"]=="TX").sum()
n_ny = (df_txny["state"]=="NY").sum()
print(f"\nTX + NY rows in primary corpus: {len(df_txny):,}")
print(f"  TX: {n_tx:,}  |  NY: {n_ny:,}")

print(f"\nFINAL TX+NY CORPUS:")
print(f"  Total rows : {len(df_txny):,}")
print(f"  TX rows    : {(df_txny['state']=='TX').sum():,} ({(df_txny['state']=='TX').sum()/len(df_txny)*100:.1f}%)")
print(f"  NY rows    : {(df_txny['state']=='NY').sum():,} ({(df_txny['state']=='NY').sum()/len(df_txny)*100:.1f}%)")

assert len(df_txny) > 1000, "Too few TX+NY rows!"
assert (df_txny["state"]=="TX").sum() > 100, "Too few TX rows!"
assert (df_txny["state"]=="NY").sum() > 100, "Too few NY rows!"

del df_primary
gc.collect()


Primary corpus after merge+dedup: 1,940,721 rows

TX + NY rows in primary corpus: 301,026
  TX: 233,868  |  NY: 67,158

FINAL TX+NY CORPUS:
  Total rows : 301,026
  TX rows    : 233,868 (77.7%)
  NY rows    : 67,158 (22.3%)


90

In [11]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — CLEAN TEXAS AND NEW YORK NLP DATASET                  ║
# ╚══════════════════════════════════════════════════════════════════╝

def clean_nlp_ds(df_raw, state_code, desc_col, name):
    """Clean NLP dataset: rename columns, drop nulls/short, clean text."""
    df = df_raw.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r"[\s\-]+","_",regex=True)
    dc = desc_col.lower().replace(" ","_")
    if dc not in df.columns:
        cands=[c for c in df.columns if any(kw in c for kw in ["desc","text","remark"])]
        if cands: dc=cands[0]; print(f"  {name}: using column '{dc}'")
        else: print(f"  {name}: NO description column!"); return pd.DataFrame()
    if dc!="description": df=df.rename(columns={dc:"description"})
    n=len(df)
    df=df.dropna(subset=["description"])
    df=df[df["description"].astype(str).str.len()>=30]
    print(f"  {name}: kept {len(df):,} of {n:,} rows")
    def ct(t):
        t=str(t); t=re.sub(r"<[^>]+>"," ",t)
        t=re.sub(r"http\S+|www\.\S+"," ",t)
        return re.sub(r"\s+"," ",t).strip()
    df["description"]=df["description"].apply(ct)
    df["state"]=state_code
    df["description_word_count"]=df["description"].apply(lambda x:len(x.split()))
    print(f"  {name}: state='{state_code}', avg {df['description_word_count'].mean():.0f} words/desc")
    return df

print("Cleaning NLP datasets ...")
df_texas_clean  =clean_nlp_ds(df_texas,  "TX",TEXAS_DESC,  "TEXAS")
df_newyork_clean=clean_nlp_ds(df_newyork,"NY",NEWYORK_DESC,"NEW YORK")
print(f"\nTEXAS   NLP: {len(df_texas_clean):,} rows")
print(f"NEW YORK NLP: {len(df_newyork_clean):,} rows")
print(f"Total NLP   : {len(df_texas_clean)+len(df_newyork_clean):,} rows")

Cleaning NLP datasets ...
  TEXAS: using column 'text'
  TEXAS: kept 12,005 of 12,137 rows
  TEXAS: state='TX', avg 141 words/desc
  NEW YORK: using column 'text'
  NEW YORK: kept 8,193 of 8,273 rows
  NEW YORK: state='NY', avg 170 words/desc

TEXAS   NLP: 12,005 rows
NEW YORK NLP: 8,193 rows
Total NLP   : 20,198 rows
